In [ ]:
# Install Transformers and Datasets libraries
!pip install -q transformers datasets accelerate

In [ ]:
import pandas as pd
from transformers import pipeline

# Load the zero-shot classification pipeline with the BART model
print("Loading the LLM model... Please wait a moment.")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("Model loaded successfully!")

In [ ]:
# 1. Define our target categories (Tags)
candidate_labels = ["Technical Issue", "Billing/Refund Inquiry", "Product Information", "Account Security"]

# 2. Create a list of mock customer support tickets
tickets = [
    "Hey, I was charged twice for my subscription this month. Can I get a refund?",
    "The Android app keeps crashing every time I try to open the checkout page. Please fix this bug.",
    "Does this wireless keyboard come with batteries included in the box, or do I need to buy them separately?",
    "Someone tried to log into my account from a different city. I need to change my password immediately.",
    "I am unable to reset my password because the verification email is not arriving in my inbox."
]

print(f"Created {len(tickets)} sample tickets for auto-tagging.")

In [ ]:
# Create empty lists to store our results
tagged_results = []

print("Processing tickets with the LLM...\n")

# Loop through each ticket and classify it
for index, ticket in enumerate(tickets, 1):
    # Perform inference
    response = classifier(ticket, candidate_labels)

    # Get the highest scoring label (the predicted tag)
    best_tag = response['labels'][0]
    confidence_score = response['scores'][0] * 100 # Convert to percentage

    # Print the live result in a clean format
    print(f"🎫 Ticket #{index}: '{ticket}'")
    print(f"🏷️ Auto-Assigned Tag: {best_tag} ({confidence_score:.2f}% confidence)")
    print("-" * 80)

    # Append to our results list for documentation
    tagged_results.append({
        "Ticket ID": f"TCK-{index:03d}",
        "Customer Text": ticket,
        "Assigned Tag": best_tag,
        "Confidence (%)": round(confidence_score, 2)
    })

In [ ]:
# Convert results into a beautiful DataFrame table
df_results = pd.DataFrame(tagged_results)
df_results

In [ ]:
# Step 1: Install Gradio library if not already installed
!pip install -q gradio

import gradio as gr
import torch
from transformers import pipeline

# Step 2: Initialize the zero-shot classification pipeline inside the app
print("Setting up Gradio Dashboard...")
app_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Define categories globally for the interface
tags_list = ["Technical Issue", "Billing/Refund Inquiry", "Product Information", "Account Security"]

# Step 3: Define the prediction function for the Gradio Interface
def auto_tag_ticket(ticket_text):
    if not ticket_text.strip():
        return "Please enter a customer support ticket text to process."

    # Run the model inference
    output = app_classifier(ticket_text, tags_list)

    # Extract predicted labels and their corresponding confidence scores
    predicted_labels = output['labels']
    scores = output['scores']

    # Create a dictionary matching each tag to its float probability score for Gradio
    results_dict = {predicted_labels[i]: float(scores[i]) for i in range(len(tags_list))}
    return results_dict

# Step 4: Design the User Interface Dashboard
ticket_dashboard = gr.Interface(
    fn=auto_tag_ticket,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Type or paste a customer email or support ticket here...",
        label="Customer Support Ticket Text"
    ),
    outputs=gr.Label(num_top_classes=4, label="LLM Auto-Assigned Tags"),
    title="🤖 AI Support Ticket Auto-Tagger (LLM)",
    description="This operational dashboard uses a Large Language Model (BART) to analyze incoming customer support texts and automatically route them into appropriate internal tags.",
    examples=[
        ["My premium account features are locked even though my credit card was charged yesterday."],
        ["I am receiving an Internal Server Error 500 when attempting to update my profile settings."],
        ["Is there a discount code available if I purchase 50 units of your software for enterprise deployment?"]
    ]
)

# Step 5: Launch the dashboard with a public shareable URL
ticket_dashboard.launch(share=True)